# Phase 2 EDA and Data Quality Validation

Business goal: understand hospital operations, financial performance, and data reliability before deploying AI models. This notebook moves the project from SQL reporting to Python profiling, outlier detection, and feature engineering.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
patients = pd.read_csv(ROOT / "patients.csv", parse_dates=["registration_date"])
visits = pd.read_csv(ROOT / "visits.csv", parse_dates=["visit_date"])
billing = pd.read_csv(ROOT / "billing.csv", parse_dates=["billing_date"])

df = visits.merge(patients, on="patient_id", how="left", validate="many_to_one").merge(
    billing, on="visit_id", how="left", validate="one_to_one"
)
df.shape

: 

## Dataset Shape

The combined dataset contains **25,000 visit-level records**. Each row links visit operations, patient demographics, insurance provider, and billing status.

In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if "df" not in globals():
    patients = pd.read_csv(ROOT / "patients.csv", parse_dates=["registration_date"])
    visits = pd.read_csv(ROOT / "visits.csv", parse_dates=["visit_date"])
    billing = pd.read_csv(ROOT / "billing.csv", parse_dates=["billing_date"])
    df = visits.merge(patients, on="patient_id", how="left", validate="many_to_one").merge(
        billing, on="visit_id", how="left", validate="one_to_one"
    )

df.head()


,visit_id,patient_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,age,gender,city,insurance_provider,chronic_flag,registration_date,bill_id,billed_amount,approved_amount,claim_status,payment_days,billing_date
0,1,756,2025-10-18,Cardiology,ER,3.48,Low,169,90,M,Bangalore,CareOne,1,2025-08-14,1,23577.37,0.00,Rejected,16.0,2025-06-18
1,2,4102,2025-04-06,Orthopedics,OPD,15.31,High,148,30,M,Hyderabad,SecureLife,1,2025-10-29,2,38178.81,38178.81,Paid,18.0,2025-10-09
2,3,2964,2025-07-13,ICU,ER,34.36,Low,153,25,F,Chennai,HealthPlus,1,2025-07-04,3,5038.97,5038.97,Paid,NaN,2025-01-20
3,4,4496,2025-11-19,Cardiology,ER,37.89,High,119,75,M,Delhi,MediCareX,0,2026-01-20,4,22813.34,22813.34,Paid,16.0,2025-12-24
4,5,1930,2025-03-29,General,ICU,16.78,Medium,118,80,M,Bangalore,HealthPlus,1,2025-03-29,5,27106.95,27106.95,Paid,14.0,2025-09-23


## Missing-Value Assessment

Critical reliability fields are `approved_amount`, `payment_days`, and `length_of_stay_hours`. Missingness in approvals affects revenue realization analysis. Missing payment days weakens payer-delay analysis. Length of stay is complete in this dataset.

In [3]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if "df" not in globals():
    patients = pd.read_csv(ROOT / "patients.csv", parse_dates=["registration_date"])
    visits = pd.read_csv(ROOT / "visits.csv", parse_dates=["visit_date"])
    billing = pd.read_csv(ROOT / "billing.csv", parse_dates=["billing_date"])
    df = visits.merge(patients, on="patient_id", how="left", validate="many_to_one").merge(
        billing, on="visit_id", how="left", validate="one_to_one"
    )

critical_fields = ["approved_amount", "payment_days", "length_of_stay_hours"]
missing_summary = pd.DataFrame({
    "field": critical_fields,
    "missing_count": [df[col].isna().sum() for col in critical_fields],
})
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(df) * 100).round(2)
missing_summary


,field,missing_count,missing_pct
0,approved_amount,1318,5.27
1,payment_days,790,3.16
2,length_of_stay_hours,0,0.00


| field                |   missing_count |   missing_pct |
|:---------------------|----------------:|--------------:|
| approved_amount      |            1318 |          5.27 |
| payment_days         |             790 |          3.16 |
| length_of_stay_hours |               0 |          0    |

![Missing values](../reports/figures/missing_values.png)

## Distribution Analysis

Visit volume is balanced across departments, while visit type and payer mix provide useful segmentation features for operations and billing models.

In [4]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if "df" not in globals():
    patients = pd.read_csv(ROOT / "patients.csv", parse_dates=["registration_date"])
    visits = pd.read_csv(ROOT / "visits.csv", parse_dates=["visit_date"])
    billing = pd.read_csv(ROOT / "billing.csv", parse_dates=["billing_date"])
    df = visits.merge(patients, on="patient_id", how="left", validate="many_to_one").merge(
        billing, on="visit_id", how="left", validate="one_to_one"
    )

for column in ["department", "visit_type", "insurance_provider", "city"]:
    display(df[column].value_counts().rename_axis(column).reset_index(name="records"))


,department,records
0,General,4228
1,ER,4220
2,Neurology,4165
3,Orthopedics,4164
4,Cardiology,4159
5,ICU,4064


,visit_type,records
0,ER,8382
1,OPD,8381
2,ICU,8237


,insurance_provider,records
0,MediCareX,6532
1,CareOne,6283
2,HealthPlus,6220
3,SecureLife,5965


,city,records
0,Hyderabad,4370
1,Pune,4221
2,Bangalore,4205
3,Mumbai,4122
4,Delhi,4107
5,Chennai,3975


![Categorical distributions](../reports/figures/categorical_distributions.png)

## Outlier Detection

Outliers are classified with IQR fences: mild outliers beyond 1.5 times IQR and extreme outliers beyond 3.0 times IQR. These flags are retained as model features instead of deleting records because high bills, delayed payments, and long stays can be valid business events.

In [5]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if "df" not in globals():
    patients = pd.read_csv(ROOT / "patients.csv", parse_dates=["registration_date"])
    visits = pd.read_csv(ROOT / "visits.csv", parse_dates=["visit_date"])
    billing = pd.read_csv(ROOT / "billing.csv", parse_dates=["billing_date"])
    df = visits.merge(patients, on="patient_id", how="left", validate="many_to_one").merge(
        billing, on="visit_id", how="left", validate="one_to_one"
    )

def classify_iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    labels = pd.Series("normal", index=series.index, dtype="object")
    labels = labels.mask(series.isna(), "missing")
    labels = labels.mask(series < q1 - 3.0 * iqr, "extreme_low")
    labels = labels.mask((series >= q1 - 3.0 * iqr) & (series < q1 - 1.5 * iqr), "mild_low")
    labels = labels.mask((series > q3 + 1.5 * iqr) & (series <= q3 + 3.0 * iqr), "mild_high")
    labels = labels.mask(series > q3 + 3.0 * iqr, "extreme_high")
    return labels

outlier_columns = ["billed_amount", "payment_days", "length_of_stay_hours"]
outlier_summary = pd.concat(
    [classify_iqr_outliers(df[col]).value_counts().rename(col) for col in outlier_columns], axis=1
).fillna(0).astype(int)
outlier_summary


,billed_amount,payment_days,length_of_stay_hours
normal,24627,23701,24744
mild_high,369,490,256
extreme_high,4,19,0
missing,0,790,0


![Outlier boxplots](../reports/figures/outlier_boxplots.png)

## Business Segment Metrics

Provider-level rejection rates show where billing reliability risk is concentrated.

| insurance_provider   |   claims |   rejection_rate |   avg_payment_days |
|:---------------------|---------:|-----------------:|-------------------:|
| SecureLife           |     5965 |           0.1569 |            13.0578 |
| MediCareX            |     6532 |           0.1525 |            12.9859 |
| HealthPlus           |     6220 |           0.1497 |            13.0703 |
| CareOne              |     6283 |           0.1487 |            13.0059 |

Department-level metrics show operational load, average length of stay, billing intensity, and high-risk visit mix.

| department   |   visits |   avg_length_of_stay_hours |   avg_billed_amount |   high_risk_rate |
|:-------------|---------:|---------------------------:|--------------------:|-----------------:|
| General      |     4228 |                    19.4349 |             20608.2 |           0.1984 |
| ER           |     4220 |                    19.535  |             21015.9 |           0.2066 |
| Neurology    |     4165 |                    19.7181 |             20962.8 |           0.2031 |
| Orthopedics  |     4164 |                    19.6627 |             21088.2 |           0.2022 |
| Cardiology   |     4159 |                    19.601  |             20695.2 |           0.1899 |
| ICU          |     4064 |                    19.3552 |             20855.7 |           0.2079 |

In [6]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if "df" not in globals():
    patients = pd.read_csv(ROOT / "patients.csv", parse_dates=["registration_date"])
    visits = pd.read_csv(ROOT / "visits.csv", parse_dates=["visit_date"])
    billing = pd.read_csv(ROOT / "billing.csv", parse_dates=["billing_date"])
    df = visits.merge(patients, on="patient_id", how="left", validate="many_to_one").merge(
        billing, on="visit_id", how="left", validate="one_to_one"
    )

provider_metrics = df.groupby("insurance_provider").agg(
    claims=("visit_id", "count"),
    rejection_rate=("claim_status", lambda s: (s == "Rejected").mean()),
    avg_payment_days=("payment_days", "mean"),
)
provider_metrics


,claims,rejection_rate,avg_payment_days
insurance_provider,,,
CareOne,6283,0.148655,13.026890
HealthPlus,6220,0.149678,13.081833
MediCareX,6532,0.152480,13.009010
SecureLife,5965,0.156915,13.078133


## Feature Engineering

The Phase 2 feature pipeline creates patient visit frequency, average length of stay per patient, provider rejection rate, days since registration, calendar features, missingness flags, imputed numeric fields, approval ratio, and outlier flags.

In [7]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.build_features import build_model_table

model_table, summaries = build_model_table()
model_table.head()


,visit_id,patient_id,bill_id,age,gender,city,insurance_provider,chronic_flag,department,visit_type,...,approval_ratio,billed_amount_outlier_class,payment_days_outlier_class,length_of_stay_hours_outlier_class,billed_amount_is_outlier,payment_days_is_outlier,length_of_stay_hours_is_outlier,visit_date,billing_date,registration_date
0,1,756,1,90,M,Bangalore,CareOne,1,Cardiology,ER,...,0.0,normal,normal,normal,0,0,0,2025-10-18,2025-06-18,2025-08-14
1,2,4102,2,30,M,Hyderabad,SecureLife,1,Orthopedics,OPD,...,1.0,normal,normal,normal,0,0,0,2025-04-06,2025-10-09,2025-10-29
2,3,2964,3,25,F,Chennai,HealthPlus,1,ICU,ER,...,1.0,normal,missing,normal,0,0,0,2025-07-13,2025-01-20,2025-07-04
3,4,4496,4,75,M,Delhi,MediCareX,0,Cardiology,ER,...,1.0,normal,normal,normal,0,0,0,2025-11-19,2025-12-24,2026-01-20
4,5,1930,5,80,M,Bangalore,HealthPlus,1,General,ICU,...,1.0,normal,normal,normal,0,0,0,2025-03-29,2025-09-23,2025-03-29


![Time features](../reports/figures/time_features.png)

## Drift and Readiness Metrics

The drift summary compares early-period and late-period visits by date. It is a lightweight pre-model monitoring check and should be refreshed when new source files arrive.

| feature                     |   early_period_mean |   late_period_mean |   pct_change |
|:----------------------------|--------------------:|-------------------:|-------------:|
| billed_amount               |          20704.2    |         21037.4    |         1.61 |
| approved_amount_filled      |          15963.3    |         16381.1    |         2.62 |
| payment_days_filled         |             13.0876 |            12.9706 |        -0.89 |
| length_of_stay_hours_filled |             19.6647 |            19.4384 |        -1.15 |
| approval_ratio              |              0.7626 |             0.7672 |         0.61 |
| provider_rejection_rate     |              0.1519 |             0.1519 |         0    |

In [9]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if "model_table" not in globals() or "summaries" not in globals():
    from scripts.build_features import build_model_table
    model_table, summaries = build_model_table()

model_table.to_csv(ROOT / "data_outputs" / "model_table.csv", index=False)
summaries["drift"].to_csv(ROOT / "data_outputs" / "drift_summary.csv", index=False)
model_table.shape


(25000, 44)

## Phase 2 Business Insights

- Approval data quality is the largest revenue-risk issue because `approved_amount` is missing in 1,318 records.
- Payment-delay reliability is weaker than operational length-of-stay reliability because `payment_days` is missing in 790 records.
- Length of stay is complete and can be used confidently for operational throughput analysis.
- Provider rejection rate and approval ratio are strong Phase 3 candidates for claim-risk or revenue-realization modeling.
- Outlier records should be retained and flagged because they represent the exact high-cost, slow-payment, and long-stay scenarios that hospital leadership needs to monitor.